<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **Space X Falcon 9 First Stage Landing Prediction**

## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia

Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches

## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`:
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


In [29]:
!pip3 install beautifulsoup4
!pip3 install requests


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

### Helper functions to process web scraped HTML table


In [42]:
def date_time(table_cells):
    """
    This function returns the date and time from the HTML table cell.
    Input: the element of a table data cell
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML table cell.
    Input: the element of a table data cell
    """
    out = ''.join([booster_version for i, booster_version in enumerate(
        table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell.
    Input: the element of a table data cell
    """
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell.
    Input: the element of a table data cell
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not(column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name

We will scrape from a snapshot of the Wikipedia page updated on **9th June 2021** for consistency.


In [43]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

### TASK 1: Request the Falcon9 Launch Wiki page from its URL


In [44]:
# Perform HTTP GET to request the Falcon9 Launch HTML page
response = requests.get(static_url, headers=headers)

# Check the response status
print(f"HTTP Status Code: {response.status_code}")
print(f"Response OK: {response.ok}")

HTTP Status Code: 403
Response OK: False


In [45]:
# Create a BeautifulSoup object from the response text content
soup = BeautifulSoup(response.text, 'html.parser')

In [46]:
# Print the page title to verify BeautifulSoup object was created properly
print(soup.title)

<title>Wikimedia Error</title>


### TASK 2: Extract all column/variable names from the HTML table header


In [47]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
html_tables = soup.find_all('table')

In [49]:
import requests
from bs4 import BeautifulSoup

column_names = []

# 1. On force la requête HTTP et la création du soup directement ici pour être sûr d'avoir les données
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(static_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# 2. On récupère les tables
html_tables = soup.find_all('table')

# Vérification de sécurité pour éviter le "out of range"
if len(html_tables) >= 3:
    first_launch_table = html_tables[2]
    
    # 3. On extrait les en-têtes
    for th in first_launch_table.find_all('th'):
        name = extract_column_from_header(th)
        if name is not None and len(name) > 0:
            column_names.append(name)
            
    print("Succès ! Voici les colonnes extraites :")
    print(column_names)
else:
    print(f"Erreur : Seulement {len(html_tables)} table(s) trouvée(s). La page ne s'est pas rechargée correctement.")

Erreur : Seulement 0 table(s) trouvée(s). La page ne s'est pas rechargée correctement.


In [50]:
print(column_names)

[]


## TASK 3: Create a data frame by parsing the launch HTML tables


In [57]:
# 1. Création du dictionnaire avec les clés extraites des en-têtes
launch_dict = dict.fromkeys(column_names)

# 2. Suppression sécurisée des colonnes de temps non pertinentes (évite les KeyError)
launch_dict.pop('Date and time ( )', None)
launch_dict.pop('Date and time ()', None)

# 3. Initialisation des listes vides pour chaque clé requise
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []

# 4. Ajout des nouvelles colonnes indispensables pour la suite
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

In [58]:
extracted_row = 0

# Extraction de chaque table wikitable
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    # Récupération des lignes de la table
    for rows in table.find_all("tr"):
        # Vérification si le premier en-tête correspond à un numéro de vol numérique
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
            
        # Récupération des éléments de la ligne (cellules td)
        row = rows.find_all('td')
        
        # Si c'est un numéro valide, on extrait et enregistre les données
        if flag:
            extracted_row += 1
            
            # Flight Number value
            launch_dict['Flight No.'].append(flight_number)
            
            datatimelist = date_time(row[0])
            
            # Date value
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            
            # Time value
            time = datatimelist[1]
            launch_dict['Time'].append(time)
              
            # Booster version
            bv = booster_version(row[1])
            if not(bv):
                bv = row[1].a.string if row[1].a else ""
            launch_dict['Version Booster'].append(bv)
            
            # Launch Site
            launch_site = row[2].a.string if row[2].a else ""
            launch_dict['Launch site'].append(launch_site)
            
            # Payload
            payload = row[3].a.string if row[3].a else ""
            launch_dict['Payload'].append(payload)
            
            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # Orbit
            orbit = row[5].a.string if row[5].a else ""
            launch_dict['Orbit'].append(orbit)
            
            # Customer
            # Sécurité renforcée si le client n'a pas de lien hypertexte <a>
            customer = row[6].a.string if row[6].a else row[6].text.strip()
            launch_dict['Customer'].append(customer)
            
            # Launch outcome
            launch_outcome = list(row[7].strings)[0] if row[7].strings else ""
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

In [59]:
# Création du dataframe à partir du dictionnaire final
df = pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

In [60]:
# 1. Nettoyage préventif des espaces autour des noms de colonnes
df.columns = df.columns.str.strip()

# Overview du dataset extrait
print("=== Dataset Info ===")
print(df.info())

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Launch outcomes ===")
if 'Launch outcome' in df.columns:
    print(df['Launch outcome'].value_counts())
else:
    print("La colonne 'Launch outcome' est absente. Colonnes dispo :", df.columns.tolist())

print("\n=== Booster landing outcomes ===")
if 'Booster landing' in df.columns:
    print(df['Booster landing'].value_counts())
else:
    print("La colonne 'Booster landing' est absente. Colonnes dispo :", df.columns.tolist())

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Flight No.       0 non-null      object
 1   Launch site      0 non-null      object
 2   Payload          0 non-null      object
 3   Payload mass     0 non-null      object
 4   Orbit            0 non-null      object
 5   Customer         0 non-null      object
 6   Launch outcome   0 non-null      object
 7   Version Booster  0 non-null      object
 8   Booster landing  0 non-null      object
 9   Date             0 non-null      object
 10  Time             0 non-null      object
dtypes: object(11)
memory usage: 132.0+ bytes
None

=== Missing values ===
Flight No.         0
Launch site        0
Payload            0
Payload mass       0
Orbit              0
Customer           0
Launch outcome     0
Version Booster    0
Booster landing    0
Date               0
Time            

In [61]:
# Exportation finale vers le fichier CSV demandé par l'exercice
df.to_csv('spacex_web_scraped.csv', index=False)
print("Fichier 'spacex_web_scraped.csv' enregistré avec succès !")

Fichier 'spacex_web_scraped.csv' enregistré avec succès !


## Authors

<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>

<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>

**Completed by:** [Hrestak33](https://github.com/Hrestak33)

Copyright © 2021 IBM Corporation. All rights reserved.
